# 第 1 章 标量衍射理论基础 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/waveprop/ch1/`。

## 圆孔 · ft · 混叠

`ch1_circle_ft_aliasing.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 1 章 =====
圆孔的二维傅里叶变换 = 夫琅禾费衍射（艾里斑），以及粗采样带来的频谱混叠。
重写自用户旧稿 Fourier_Transfer_2D.m / circle_digital.m / 二维傅里叶变换.py / 圆孔离散化.py：
  - cv2 画圆改为解析 circ(R<=D/2)，去掉 cv2 依赖；
  - 细/粗两套网格合并到一个脚本，补上与连续理论的定量核对。
用法：直接运行弹窗显示；`python ch1_circle_ft_aliasing.py <输出目录>` 则保存 PNG。
依赖：numpy + matplotlib。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (注册 3d projection)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ===== 1. 参数 =====
D = 100e-6          # 圆孔直径 100 um（对应原书 circ 函数，Appendix A）
wvl = 532e-9        # 波长 532 nm

# --- A. 细网格：逼近连续圆孔 ---
N_f = 512
dx_f = 1e-6                       # 1 um 像素，圆孔直径占 100 像素
x_f = (np.arange(N_f) - N_f // 2 + 0.5) * dx_f
X_f, Y_f = np.meshgrid(x_f, x_f)
U_f = (np.sqrt(X_f ** 2 + Y_f ** 2) <= D / 2).astype(float)   # circ(2r/D)

# --- B. 粗网格：同一个圆孔，像素放大 4 倍 ---
N_c = 128
dx_c = 4e-6                       # 4 um 像素，圆孔直径只占 25 像素 -> 边缘锯齿
x_c = (np.arange(N_c) - N_c // 2 + 0.5) * dx_c
X_c, Y_c = np.meshgrid(x_c, x_c)
U_c = (np.sqrt(X_c ** 2 + Y_c ** 2) <= D / 2).astype(float)

# ===== 2. 居中二维傅里叶变换（ft2，含面积元 dx^2）=====
def ft2(g, delta):
    """G(fx,fy) = FFT2(g) 的中心化排列 * delta^2，近似连续傅里叶变换。
    坐标约定 x = (n - N/2 + 1/2) delta（胞元中心），频率轴 fx = (k - N/2)/(N delta)。
    MATLAB 同名实现见笔记，注意 fftshift 在 MATLAB 里对每维同样适用。"""
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(g))) * delta ** 2

G_f = ft2(U_f, dx_f)
I_f = np.abs(G_f) ** 2
I_f /= I_f.max()
f_f = (np.arange(N_f) - N_f // 2) / (N_f * dx_f)          # 频率轴 (1/m)

G_c = ft2(U_c, dx_c)
I_c = np.abs(G_c) ** 2
I_c /= I_c.max()
f_c = (np.arange(N_c) - N_c // 2) / (N_c * dx_c)

# ===== 3. 数值验证 =====
print('===== 圆孔傅里叶变换数值验证 =====')
# (1) 直流值：FT{circ(2r/D)}(0) = 圆孔面积 = pi D^2 / 4
dc_th = np.pi * D ** 2 / 4
dc_num = np.abs(G_f[N_f // 2, N_f // 2])
print(f'(1) 直流分量理论 pi*D^2/4 = {dc_th:.4e}，数值 = {dc_num:.4e}，相对偏差 {abs(dc_num-dc_th)/dc_th:.2e}')

# (2) 艾里斑第一暗环：J1 零点 3.8317 -> f = 3.8317/(pi*D)；三点抛物线细化极小位置
f1_th = 3.8317 / (np.pi * D)
row = I_f[N_f // 2, :]                       # 沿 fx 轴（fy=0）的剖面
half = N_f // 2 + 1                          # f>0 半轴
rings = [i for i in range(half, N_f - 1) if row[i] < row[i - 1] and row[i] < row[i + 1]]
i0 = rings[0]
df_bin = f_f[1] - f_f[0]
den = row[i0 - 1] - 2 * row[i0] + row[i0 + 1]
f1_num = f_f[i0] + 0.5 * df_bin * (row[i0 - 1] - row[i0 + 1]) / (den + 1e-30)
print(f'(2) 第一暗环理论 f = {f1_th/1e3:.3f} mm^-1，数值（细化）= {f1_num/1e3:.3f} mm^-1，'
      f'相对偏差 {abs(f1_num-f1_th)/f1_th:.2e}')

# (3) 粗网格的两类失真（沿 fx 剖面对比细网格）：
#     像素化边缘把光散射进艾里暗环（低频段暗环被"填亮"）；
#     频谱以 1/dx_c 为周期复制，在近奈奎斯特端（|f|>100 mm^-1）出现混叠尾巴。
fx_c_mm = f_c * 1e-3
I_f_interp = np.interp(fx_c_mm, f_f * 1e-3, row / row[half])   # 细网格归一化剖面
prof_c = I_c[N_c // 2, :] / I_c[N_c // 2, N_c // 2]
i_ring = int(np.argmin(np.abs(fx_c_mm - f1_th * 1e-3)))
fill = prof_c[i_ring] / max(I_f_interp[i_ring], 1e-12)
high = np.abs(fx_c_mm) > 100
err_high = np.max(np.abs(prof_c[high] - I_f_interp[high]))
print(f'(3) 第一暗环处 粗/细强度比 = {fill:.1f}（像素化把暗环填亮，细网格处接近 0）；'
      f'|f|>100 mm^-1 频谱复制尾巴最大绝对偏差 = {err_high:.2e}')

# ===== 4. 绘图 =====
def make_figures():
    fig = plt.figure(figsize=(15, 9.5))

    ax1 = fig.add_subplot(2, 3, 1)
    ax1.imshow(U_f, extent=[x_f[0] * 1e6, x_f[-1] * 1e6] * 2, cmap='gray', origin='lower')
    ax1.set_title('(a) 细网格圆孔（连续近似）')
    ax1.set_xlabel('x (um)'); ax1.set_ylabel('y (um)')

    ax2 = fig.add_subplot(2, 3, 2)
    im2 = ax2.imshow(np.log10(I_f + 1e-6), extent=[f_f[0] * 1e-3, f_f[-1] * 1e-3] * 2,
                     cmap='gray', vmin=-5, vmax=0, origin='lower')
    fig.colorbar(im2, ax=ax2, shrink=0.85).set_label('log$_{10}$(I/I$_{max}$)')
    ax2.set_title('(b) 艾里斑频谱（对数尺度）')
    ax2.set_xlabel('$f_x$ (mm$^{-1}$)'); ax2.set_ylabel('$f_y$ (mm$^{-1}$)')

    ax3 = fig.add_subplot(2, 3, 3, projection='3d')
    s = 4                                     # 3D 面抽稀，仅中心 1/4 区域
    c = slice(N_f // 2 - 32 * s, N_f // 2 + 32 * s, s)
    FXg, FYg = np.meshgrid(f_f[c] * 1e-3, f_f[c] * 1e-3)
    ax3.plot_surface(FXg, FYg, I_f[c, c], cmap='jet', edgecolor='none')
    ax3.set_title('(c) 艾里斑 3D 光强')
    ax3.set_xlabel('$f_x$ (mm$^{-1}$)'); ax3.set_ylabel('$f_y$ (mm$^{-1}$)')

    ax4 = fig.add_subplot(2, 3, 4)
    ax4.imshow(U_c, extent=[x_c[0] * 1e6, x_c[-1] * 1e6] * 2, cmap='gray', origin='lower')
    ax4.set_title('(d) 粗网格圆孔（边缘锯齿）')
    ax4.set_xlabel('x (um)'); ax4.set_ylabel('y (um)')

    ax5 = fig.add_subplot(2, 3, 5)
    im5 = ax5.imshow(np.log10(I_c + 1e-6), extent=[f_c[0] * 1e-3, f_c[-1] * 1e-3] * 2,
                     cmap='gray', vmin=-5, vmax=0, origin='lower')
    fig.colorbar(im5, ax=ax5, shrink=0.85).set_label('log$_{10}$(I/I$_{max}$)')
    ax5.set_title('(e) 粗网格频谱：高频端出现混叠')
    ax5.set_xlabel('$f_x$ (mm$^{-1}$)'); ax5.set_ylabel('$f_y$ (mm$^{-1}$)')

    ax6 = fig.add_subplot(2, 3, 6)
    ax6.plot(f_f * 1e-3, row / row[half], 'b-', lw=1.2, label='细网格 (dx=1 um)')
    ax6.plot(fx_c_mm, prof_c, 'r.-', lw=0.8, ms=4, label='粗网格 (dx=4 um)')
    ax6.axvline(f1_th * 1e-3, color='g', ls=':', lw=1.2,
                label='理论第一暗环 12.2 mm$^{-1}$')
    ax6.set_xlim(-60, 60); ax6.set_ylim(1e-5, 1.1)
    ax6.set_yscale('log')
    ax6.set_xlabel('$f_x$ (mm$^{-1}$)'); ax6.set_ylabel('归一化光强')
    ax6.set_title('沿 $f_x$ 剖面对比（对数纵轴）')
    ax6.legend(); ax6.grid(alpha=0.3, which='both')

    fig.tight_layout()
    return fig


if __name__ == '__main__':
    if len(sys.argv) > 1:
        outdir = sys.argv[1]
        os.makedirs(outdir, exist_ok=True)
        fig = make_figures()
        fig.savefig(os.path.join(outdir, 'circle_ft_aliasing.png'), dpi=150)
        print('已保存：circle_ft_aliasing.png ->', outdir)
    else:
        make_figures()
        plt.show()


## DFT · 三步 · 法

`ch1_dft_three_steps.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 1 章（衔接第 2 章主题）=====
离散傅里叶变换的三步演化：空间采样 -> 有限截断 -> 频域采样。
演示 g(x)=exp(-a|x|) 从连续傅里叶变换到 DFT 的每一步在两个域里各损失/改变了什么。
重写自用户旧稿 Alleviating_Effects_of_Discretization.m / 采样结果.py，参数一致：
a=10, delta=0.0375, L=0.6, N=16。
用法：直接运行弹窗显示；`python ch1_dft_three_steps.py <输出目录>` 则保存 PNG。
依赖：numpy + matplotlib。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ===== 1. 参数 =====
a = 10.0                    # 指数衰减系数
delta = 0.0375              # 空间采样间隔（MATLAB: delta；Python 避开关键字取名 delta）
L = 0.6                     # 截断窗口宽度
N = int(round(L / delta))   # 采样点数 N = 16

# ===== 2. 连续参考 =====
N_cont = 2000
x_cont = np.linspace(-1.5, 1.5, N_cont)
g_cont = np.exp(-a * np.abs(x_cont))
f_cont = np.linspace(-25.0, 25.0, N_cont)
G_cont = 2 * a / (a ** 2 + (2 * np.pi * f_cont) ** 2)   # FT{exp(-a|x|)} 解析式

# ===== 3. 三步演化 =====
# --- 步骤 1：仅采样（空间域乘 comb，频域卷积 comb -> 频谱以 1/delta 为周期复制）---
x_samp = np.arange(-1.5, 1.5 + delta / 2, delta)
g_samp = np.exp(-a * np.abs(x_samp))
G_repl = np.zeros_like(f_cont)
for p in range(-2, 3):                       # 叠 5 个相邻副本示意混叠
    G_repl += 2 * a / (a ** 2 + (2 * np.pi * (f_cont - p / delta)) ** 2)

# --- 步骤 2：有限截断（空间域乘 rect(x/L)，频域卷积 L*sinc(L f) -> 波纹）---
x_trunc = (np.arange(N) - N // 2) * delta
g_trunc = np.exp(-a * np.abs(x_trunc))
M = 4096                                     # 高倍补零，模拟"连续的带波纹频谱"
g_pad = np.zeros(M)
g_pad[M // 2 - N // 2: M // 2 + N // 2] = g_trunc
G_rip = np.abs(np.fft.fftshift(np.fft.fft(np.fft.fftshift(g_pad)))) * delta
f_dense = (np.arange(M) - M // 2) / (M * delta)

# --- 步骤 3：频域采样（最终 DFT：频域乘 comb(L f)，空间域周期延拓）---
df = 1.0 / (N * delta)
f_dft = (np.arange(N) - N // 2) * df
G_dft = np.abs(np.fft.fftshift(np.fft.fft(np.fft.fftshift(g_trunc)))) * delta

# ===== 4. 数值验证 =====
print('===== DFT 三步演化数值验证 =====')
# (1) 混叠量：步骤 1 的复制谱在 f=1/(2 delta) 处相对连续谱的抬升
f_nyq = 1 / (2 * delta)
ali = (G_repl[np.argmin(np.abs(f_cont - f_nyq))] / G_cont[np.argmin(np.abs(f_cont - f_nyq))])
print(f'(1) 奈奎斯特频率 {f_nyq:.2f} 处 复制谱/连续谱 = {ali:.3f}（采样间隙造成的频谱抬升）')

# (2) 最终 DFT vs 连续谱：在 DFT 频点 f_m = m/(N delta) 上比较
G_dft_bin = np.abs(np.fft.fftshift(np.fft.fft(np.fft.fftshift(g_trunc)))) * delta
G_ana_bin = 2 * a / (a ** 2 + (2 * np.pi * f_dft) ** 2)
relerr = np.max(np.abs(G_dft_bin - G_ana_bin)) / G_ana_bin.max()
print(f'(2) DFT 谱线 vs 连续解析谱（同频点）最大偏差 = {relerr:.2e}'
      f'（来源：截断丢尾 + 邻副本混叠，无法归零）')

# (3) Parseval 能量守恒（DFT 对自身的精确性质，检查实现无误）
lhs = np.sum(g_trunc ** 2) * delta
rhs = np.sum((np.fft.fft(np.fft.fftshift(g_trunc)) * delta) ** 2).real * df
print(f'(3) Parseval：sum|g|^2*delta = {lhs:.6e}，sum|G|^2*df = {rhs:.6e}，'
      f'相对偏差 {abs(lhs-rhs)/lhs:.2e}')

# ===== 5. 绘图 =====
def make_figures():
    fig, axes = plt.subplots(3, 2, figsize=(12, 10))

    axes[0, 0].plot(x_cont, g_cont, color='0.75', lw=1.5)
    axes[0, 0].stem(x_samp, g_samp, linefmt='r-', markerfmt=' ', basefmt='r-')
    axes[0, 0].set_xlim(-1.2, 1.2); axes[0, 0].set_ylim(0, 1.1)
    axes[0, 0].set_title('步骤 1 空间采样：g(x)·comb(x/δ)（无限采样列）')
    axes[0, 0].set_ylabel('g(x)')

    axes[0, 1].plot(f_cont, G_cont, color='0.75', lw=1.5)
    axes[0, 1].plot(f_cont, G_repl, 'r-', lw=2)
    axes[0, 1].set_xlim(-20, 20); axes[0, 1].set_ylim(0, 0.25)
    axes[0, 1].set_title('步骤 1 频域：卷积 comb(δf) → 频谱周期复制（混叠）')
    axes[0, 1].set_ylabel('G(f)')

    out = np.abs(x_samp) >= L / 2
    axes[1, 0].vlines(x_samp[out], 0, g_samp[out], color='0.75', lw=1.2)  # 被丢弃的采样点
    axes[1, 0].stem(x_trunc, g_trunc, linefmt='b-', markerfmt='bo', basefmt='b-')
    axes[1, 0].set_xlim(-1.2, 1.2); axes[1, 0].set_ylim(0, 1.1)
    axes[1, 0].set_title('步骤 2 有限截断：·rect(x/L)（窗口外采样点丢弃）')
    axes[1, 0].set_ylabel('g(x)')

    axes[1, 1].plot(f_cont, G_repl, color='0.75', lw=1.5)
    axes[1, 1].plot(f_dense, G_rip, 'b-', lw=2)
    axes[1, 1].set_xlim(-20, 20); axes[1, 1].set_ylim(0, 0.25)
    axes[1, 1].set_title('步骤 2 频域：卷积 L·sinc(Lf) → 吉布斯波纹')
    axes[1, 1].set_ylabel('G(f)')

    for off in (-L, 0, L):
        axes[2, 0].plot(x_cont + off, g_cont, '--', color='0.75', lw=1)
    axes[2, 0].stem(x_trunc, g_trunc, linefmt='g-', markerfmt='go', basefmt='g-')
    axes[2, 0].set_xlim(-1.2, 1.2); axes[2, 0].set_ylim(0, 1.1)
    axes[2, 0].set_title('步骤 3 频域采样：空间域卷积 comb(x/L) → 周期延拓')
    axes[2, 0].set_ylabel(r'$\tilde{g}(x)$')

    axes[2, 1].plot(f_dense, G_rip, color='0.75', lw=1.5)
    axes[2, 1].stem(f_dft, G_dft, linefmt='g-', markerfmt='go', basefmt='g-')
    axes[2, 1].set_xlim(-20, 20); axes[2, 1].set_ylim(0, 0.25)
    axes[2, 1].set_title('步骤 3 频域：×comb(Lf) → 最终 N 点 DFT 谱线')
    axes[2, 1].set_ylabel(r'$\tilde{G}(f)$')

    for ax in axes.ravel():
        ax.grid(alpha=0.25)
    fig.tight_layout()
    return fig


if __name__ == '__main__':
    if len(sys.argv) > 1:
        outdir = sys.argv[1]
        os.makedirs(outdir, exist_ok=True)
        fig = make_figures()
        fig.savefig(os.path.join(outdir, 'dft_three_steps.png'), dpi=150)
        print('已保存：dft_three_steps.png ->', outdir)
    else:
        make_figures()
        plt.show()


## 几何光学 · vs · 波动光学对比

`ch1_geom_vs_wave.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 1 章 =====
几何光学 vs 波动光学：不透明圆盘（水滴模型）挡光后的光强分布。
几何光学给出"绝对阴影"；波动光学（第 1 章的菲涅耳衍射积分，式 1.57）给出
阴影边缘的衍射条纹，以及阴影中心残留的亮斑（泊松斑）。
重写自用户旧稿 几何光学和波动光学对比.py：旧版用 1-(D/λz)·sinc 的启发式公式示意，
本版改为按式 (1.57) 的传递函数法真实传播，物理量纲正确。
用法：直接运行弹窗显示；`python ch1_geom_vs_wave.py <输出目录>` 则保存 PNG。
依赖：numpy + matplotlib。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ===== 1. 参数 =====
N = 1024
dx = 2e-6            # 2 um 像素
wvl = 532e-9
k = 2 * np.pi / wvl
z = 5e-3             # 传播 5 mm（传递函数法窗口内：z < 4 dx^2/λ ≈ 30 mm）

x = (np.arange(N) - N // 2 + 0.5) * dx
X, Y = np.meshgrid(x, x)
R = np.sqrt(X ** 2 + Y ** 2)

D = 100e-6           # 圆盘（水滴）直径 100 um

# ===== 2. 两个模型 =====
# 几何光学：圆盘内光强为 0，其余为 1（绝对阴影）
I_geom = (R > D / 2).astype(float)

# 波动光学：U1 = 1 - circ（平面波减圆盘），用式(1.57)的 TF 形式传播
U1 = (R > D / 2).astype(float)
fx = (np.arange(N) - N // 2) / (N * dx)
FX, FY = np.meshgrid(fx, fx)
H = np.exp(1j * k * z) * np.exp(-1j * np.pi * wvl * z * (FX ** 2 + FY ** 2))
U_ft = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(U1)))
U2 = np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(U_ft * H)))
I_wave = np.abs(U2) ** 2

# ===== 3. 数值验证 =====
print('===== 几何 vs 波动数值验证 =====')
ctr = N // 2
# (1) 泊松斑：几何光学预言盘心 I=0；波动光学预言轴上振幅恢复到入射值（I=1）。
#     网格离散下泊松斑尺度 sqrt(λz)=1.6 um 小于像素，中心值是它的像素平均残影。
print(f'(1) 盘心光强：几何 = 0，波动数值 = {I_wave[ctr, ctr]:.3f}（理想连续值 1）')
# (2) 阴影边缘：几何光学的跳变 vs 波动光学的条纹（中心行剖面）
row_g, row_w = I_geom[ctr, :], I_wave[ctr, :]
out_sh = (np.abs(x) > D / 2) & (np.abs(x) < D / 2 + 50e-6)   # 阴影外 50 um 环带
print(f'(2) 阴影外 50 um 环带内 波动光强范围 = [{row_w[out_sh].min():.3f}, '
      f'{row_w[out_sh].max():.3f}]（几何恒为 1，衍射产生明暗条纹）')
# (3) 能量守恒：TF 传播前后总能量（|U|^2 求和）应基本不变
print(f'(3) 传播前 sum|U1|^2 = {np.sum(U1**2):.1f}，传播后 sum|U2|^2 = {np.sum(I_wave):.1f}')

# ===== 4. 绘图 =====
def make_figures():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    im = ax1.imshow(I_wave, extent=[x[0] * 1e6, x[-1] * 1e6] * 2, cmap='inferno',
                    vmin=0, vmax=1, origin='lower')
    fig.colorbar(im, ax=ax1, shrink=0.85).set_label('相对光强')
    ax1.add_patch(plt.Circle((0, 0), D / 2 * 1e6, fill=False, color='w', ls='--', lw=0.8))
    ax1.set_xlim(-200, 200); ax1.set_ylim(-200, 200)
    ax1.set_title(f'波动光学：圆盘后 z={z*1e3:.0f} mm 的光强分布')
    ax1.set_xlabel('x (um)'); ax1.set_ylabel('y (um)')

    ax2.plot(x * 1e6, row_g, 'r--', lw=1.5, label='几何光学（绝对阴影）')
    ax2.plot(x * 1e6, row_w, 'b-', lw=1.5, label='波动光学（菲涅耳衍射）')
    ax2.axvspan(-D / 2 * 1e6, D / 2 * 1e6, color='gray', alpha=0.12)
    ax2.set_xlim(-150, 150); ax2.set_ylim(-0.05, 1.15)
    ax2.set_xlabel('x (um)'); ax2.set_ylabel('相对光强')
    ax2.set_title('中心行剖面：两种模型的对比')
    ax2.legend(); ax2.grid(alpha=0.3)

    fig.tight_layout()
    return fig


if __name__ == '__main__':
    if len(sys.argv) > 1:
        outdir = sys.argv[1]
        os.makedirs(outdir, exist_ok=True)
        fig = make_figures()
        fig.savefig(os.path.join(outdir, 'geom_vs_wave.png'), dpi=150)
        print('已保存：geom_vs_wave.png ->', outdir)
    else:
        make_figures()
        plt.show()


## 双缝干涉 · diffraction

`ch1_slit_diffraction.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 1 章 =====
单缝衍射 / 双缝干涉：菲涅耳近场 + 夫琅禾费远场
对应原书式 (1.57)（菲涅耳衍射积分）与式 (1.67)、(1.71)（夫琅禾费衍射）。
重写自用户旧稿 InterferenceDiffraction.m / 双缝干涉和单缝衍射.py：
  - 旧 MATLAB 版 propFresnelTF 漏写了 fft2（H 根本没乘到频谱上），本版已修正；
  - OpenCV 画缝改为解析 rect 组合，去掉 cv2 依赖；
  - 统一采用"胞元中心"采样约定（x 偏移半像素）：40 um 矩形缝恰占 20 个胞元，
    离散谱零点严格落在连续理论位置 m*lambda*z/Dx 上（节点约定会偏差 1+1/M）。
用法：直接运行弹窗显示；`python ch1_slit_diffraction.py <输出目录>` 则保存 PNG。
依赖：numpy + matplotlib（不需要 scipy / cv2）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ===== 1. 物理与网格参数 =====
N = 1024            # 网格点数（N x N）
dx = 2e-6           # 采样间隔 2 um（MATLAB 数组从 1 起、Python 从 0 起，坐标式一致）
wvl = 532e-9        # 波长 532 nm（变量名避开 Python 关键字 lambda）
k = 2 * np.pi / wvl

L = N * dx                                    # 网格物理宽度 2.048 mm
x1 = (np.arange(N) - N // 2 + 0.5) * dx       # 胞元中心坐标：-L/2+dx/2 ... L/2-dx/2
X1, Y1 = np.meshgrid(x1, x1)                  # numpy meshgrid 与 MATLAB 同名同义

# 单缝：宽 Dx、高 Dy；双缝：中心距 dx_slit（> Dx，两缝不重叠）
Dx = 40e-6
Dy = 400e-6
dx_slit = 120e-6

U_single = ((np.abs(X1) <= Dx / 2) & (np.abs(Y1) <= Dy / 2)).astype(float)
# 双缝 = 两个 rect 之和：缝中心分别在 x = ±dx_slit/2（相干叠加，直接相加）
U_double = ((np.abs(np.abs(X1) - dx_slit / 2) <= Dx / 2)
            & (np.abs(Y1) <= Dy / 2)).astype(float)

z_near = 5e-3      # 近场观察距离 5 mm（菲涅耳区）
z_far = 80e-3      # 远场观察距离 80 mm（夫琅禾费区）


# ===== 2. 传播算子 =====
def prop_fresnel_tf(U_in, delta, wavelength, z):
    """菲涅耳衍射·传递函数法（TF）：U2 = IFT2{ FT2{U1} * H }，
    H(fx,fy) = exp(i k z) exp(-i pi lambda z (fx^2+fy^2))。
    传递函数法适用条件：pi*lambda*z/(4 dx^2) < pi，即 z < 4 dx^2/lambda（本例 30 mm）。
    频率轴取 fx = (n - N/2)/(N dx)，与胞元中心 x 配对（半像素相移不改变光强）。
    """
    M2, N2 = U_in.shape
    fx = (np.arange(N2) - N2 // 2) / (N2 * delta)
    fy = (np.arange(M2) - M2 // 2) / (M2 * delta)
    FX, FY = np.meshgrid(fx, fy)
    H = np.exp(1j * k * z) * np.exp(-1j * np.pi * wavelength * z * (FX ** 2 + FY ** 2))
    U_ft = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(U_in)))   # 中心化谱
    U_out = np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(U_ft * H)))
    return U_out


def fraunhofer_fft(U_in, delta, wavelength, z):
    """夫琅禾费衍射（式 1.67）：U2 = e^{ikz} e^{ik(x2^2+y2^2)/(2z)} /(i lambda z) * FT2{U1}，
    观察面坐标由频率缩放得到 x2 = lambda z * fx。返回 (U2, X2, Y2)。
    """
    M2, N2 = U_in.shape
    fx = (np.arange(N2) - N2 // 2) / (N2 * delta)
    fy = (np.arange(M2) - M2 // 2) / (M2 * delta)
    FX, FY = np.meshgrid(fx, fy)
    x2 = wavelength * z * FX          # 远场屏幕真实坐标（米）
    y2 = wavelength * z * FY
    G = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(U_in))) * delta ** 2   # 连续 FT 的离散近似
    U_out = np.exp(1j * k * z) * np.exp(1j * k * (x2 ** 2 + y2 ** 2) / (2 * z)) \
        / (1j * wavelength * z) * G
    return U_out, x2, y2


# ===== 3. 计算 =====
I_single_fres = np.abs(prop_fresnel_tf(U_single, dx, wvl, z_near)) ** 2
I_double_fres = np.abs(prop_fresnel_tf(U_double, dx, wvl, z_near)) ** 2

U_s_far, X2, Y2 = fraunhofer_fft(U_single, dx, wvl, z_far)
U_d_far, _, _ = fraunhofer_fft(U_double, dx, wvl, z_far)
I_single_far = np.abs(U_s_far) ** 2
I_double_far = np.abs(U_d_far) ** 2

x1_um = x1 * 1e6          # 源平面坐标（um）
x2_um = X2[0, :] * 1e6    # 远场横向坐标（um）

# ===== 4. 数值验证（运行后核对打印结果） =====
def padded_far_profile(U_in):
    """4 倍补零后再取远场 |U| 剖面：把极值定位精度提高约 4 倍。
    补零在 fftshift 后的一维序列上进行：正坐标样本（前半段）放头部，
    负坐标样本（后半段）接到尾部，保证采样列在 4N 长度上仍等距连续、
    相位参考点（x≈0）不动，否则频谱会出现虚假调制。
    """
    Np = 4 * N
    h = np.fft.fftshift(U_in[N // 2, :])           # y=0 行
    hp = np.zeros(Np)
    hp[:N // 2] = h[:N // 2]                       # x = +dx/2 ... +L/2-dx/2
    hp[-(N // 2):] = h[N // 2:]                    # x = -L/2+dx/2 ... -dx/2（环形接回尾部）
    F = np.fft.fft(hp) * dx                        # 连续 FT 的离散近似（一维）
    f = np.arange(Np) / (Np * dx)                  # k >= 0 半轴频率
    f[f > 1 / (2 * dx)] -= 1 / dx                  # 折回负频率
    return np.abs(F) ** 2, f * wvl * z_far * 1e6   # (I, x2/um)


print('===== 夫琅禾费衍射数值验证（4 倍补零，y2 = 0 剖面）=====')
I_pad_d, x2_pad = padded_far_profile(U_double)
I_pad_s, _ = padded_far_profile(U_single)

# (1) 左右对称性：在镜像点 -x2 处插值取值再比较。
# 注意：偶数 N 的中心化频率轴是 -N/2...N/2-1 个 bin，本身左右差一个 bin，
# 直接 I 与 I[::-1] 翻转比较会错位一个 bin，必须插值到 -x2。
prof_d = I_double_far[N // 2, :]
mirror = np.interp(-x2_um, x2_um, prof_d)
sym = np.max(np.abs(prof_d - mirror)) / prof_d.max()
print(f'(1) 双缝远场剖面左右对称性  max|I(x2)-I(-x2)|/max(I) = {sym:.2e}')

# (2) 单缝暗纹：sinc 零点（严格）理论位置 x2 = m * lambda*z/Dx
dark_th = np.arange(1, 8) * wvl * z_far / Dx * 1e6
win = (x2_pad > 0) & (x2_pad < dark_th[-1] * 1.1)
xm, pm = x2_pad[win], I_pad_s[win]
dark_num = np.array([xm[i] for i in range(1, len(pm) - 1)
                     if pm[i] < pm[i - 1] and pm[i] < pm[i + 1]])
rel_dark = np.max(np.abs(dark_num[:len(dark_th)] - dark_th) / dark_th)
print(f'(2) 单缝暗纹理论 (um) = {np.round(dark_th, 1)}')
print(f'    数值极小   (um) = {np.round(dark_num[:len(dark_th)], 1)}  最大相对偏差 {rel_dark:.2e}')

# (3) 双缝：暗纹零点严格位于 (n+1/2)*lambda*z/dx_slit 与包络零点 m*lambda*z/Dx；
#     亮纹极大受单缝包络斜坡影响，会从 cos^2 极大 n*lambda*z/dx_slit 向中心偏移
#     （第一亮纹约 341 um 而非 354.7 um），故应与式 (1.71) 解析剖面的极大对比。
cos_zero_th = (np.arange(6) + 0.5) * wvl * z_far / dx_slit * 1e6
env_zero_th = np.arange(1, 7) * wvl * z_far / Dx * 1e6
sub = (x2_pad > 0) & (x2_pad < 6.5 * wvl * z_far / dx_slit * 1e6)
xd, pd = x2_pad[sub], I_pad_d[sub]     # xd/pd 是子集，之后一律用子集索引
# --- 3a 暗纹：取数值剖面在理论零点处的值，与邻近干涉条纹峰值之比 ---
frac = []
sp = wvl * z_far / dx_slit * 1e6          # cos^2 因子的条纹间距 (um)
for z0 in np.concatenate([cos_zero_th[:3], env_zero_th[:2]]):
    i0 = int(np.argmin(np.abs(xd - z0)))
    win = np.abs(xd - z0) < 0.9 * sp      # 邻近条纹峰：零点两侧一个条纹周期内取最大
    frac.append(pd[i0] / pd[win].max())
print(f'(3a) 双缝暗纹深度：理论零点处 I/邻近条纹峰 = {np.round(np.array(frac), 5)}（全部 << 1）')
# --- 3b 亮纹：解析式在细网格上找极大；数值谱直接取 bin 极大 ---
x2_fine = np.arange(0.0, 6.2 * wvl * z_far / dx_slit * 1e6, 0.2)
pa_fine = (np.cos(np.pi * dx_slit * (x2_fine * 1e-6) / (wvl * z_far)) ** 2
           * np.sinc(Dx * (x2_fine * 1e-6) / (wvl * z_far)) ** 2)
bright_ana = np.array([x2_fine[i] for i in range(1, len(pa_fine) - 1)
                       if pa_fine[i] > pa_fine[i - 1] and pa_fine[i] > pa_fine[i + 1]])
bright_num = np.array([xd[i] for i in range(1, len(pd) - 1)
                       if pd[i] > pd[i - 1] and pd[i] > pd[i + 1]])
nb = min(len(bright_num), len(bright_ana))
rel_bright = np.max(np.abs(bright_num[:nb] - bright_ana[:nb]) / bright_ana[:nb])
print(f'(3b) 亮纹 cos^2 极大（朴素）(um) = {np.round(np.arange(1, 7) * sp, 1)}')
print(f'     解析式(1.71)极大     (um) = {np.round(bright_ana[:nb], 1)}  <- 被包络拉向中心')
bin_um = wvl * z_far / (4 * N * dx) * 1e6
print(f'     数值极大             (um) = {np.round(bright_num[:nb], 1)}  （bin 距 {bin_um:.1f} um 量化）')
print(f'     数值 vs 解析极大最大相对偏差 = {rel_bright:.2e}')
miss = [n for n in (3, 6) if np.min(np.abs(bright_num - n * sp)) > 30]
print(f'     缺级检验：n=3,6 处无极大 -> 缺失级次 = {miss}（理论预期 dx_slit/Dx=3 -> [3, 6]）')

# (4) 与式 (1.71) 解析解对比：|U| 沿 y2 = 0，取显示窗口 ±800 um（未补零原网格）
U_ana = 2 * Dx * Dy / (wvl * z_far) * np.cos(np.pi * dx_slit * X2 / (wvl * z_far)) \
    * np.sinc(Dx * X2 / (wvl * z_far)) * np.sinc(Dy * Y2 / (wvl * z_far))
win4 = np.abs(x2_um) <= 800
err = np.max(np.abs(np.abs(U_d_far[N // 2, win4]) - np.abs(U_ana[N // 2, win4]))) \
    / np.abs(U_d_far[N // 2, :]).max()
print(f'(4) FFT 数值解 vs 式(1.71)解析解（±800 um 内）  max||U|-|U_ana||/max|U| = {err:.2e}')

# ===== 5. 绘图 =====
def make_figures():
    fig1, axes = plt.subplots(2, 2, figsize=(11, 9))
    panels = [
        (axes[0, 0], I_single_fres, x1_um, f'单缝·菲涅耳近场 z={z_near*1e3:.0f} mm'),
        (axes[0, 1], I_single_far, x2_um, f'单缝·夫琅禾费远场 z={z_far*1e3:.0f} mm'),
        (axes[1, 0], I_double_fres, x1_um, f'双缝·菲涅耳近场 z={z_near*1e3:.0f} mm'),
        (axes[1, 1], I_double_far, x2_um, f'双缝·夫琅禾费干涉 z={z_far*1e3:.0f} mm'),
    ]
    for ax, I, xc, title in panels:
        im = ax.imshow(I, extent=[xc[0], xc[-1], xc[0], xc[-1]], cmap='gray',
                       vmin=0, vmax=0.02 * I.max(), origin='lower')
        ax.set_title(title)   # vmax 截到 2%：压掉 0 级亮斑，突出条纹
        ax.set_xlabel('x (um)')
        ax.set_ylabel('y (um)')
        ax.set_xlim(-800, 800)
        ax.set_ylim(-800, 800)
        fig1.colorbar(im, ax=ax, shrink=0.85)
    fig1.tight_layout()

    fig2, ax = plt.subplots(figsize=(10, 5))
    norm = I_pad_d.max()
    ax.plot(x2_pad, I_pad_d / norm, 'b-', lw=1.2, label='双缝 FFT 数值解（4 倍补零）')
    env = np.sinc(Dx * (x2_pad * 1e-6) / (wvl * z_far)) ** 2    # 单缝包络 sinc^2
    ax.plot(x2_pad, env, 'g--', lw=1.2, label='单缝包络 sinc$^2$($D_x x_2/\\lambda z$)')
    ax.set_xlim(-800, 800)
    ax.set_ylim(-0.02, 1.05)
    ax.set_xlabel('远场坐标 $x_2$ (um)')
    ax.set_ylabel('归一化光强')
    ax.set_title('双缝夫琅禾费干涉：$\\Delta x = 3D_x$ → ±3 级缺级')
    ax.legend()
    ax.grid(alpha=0.3)
    fig2.tight_layout()
    return fig1, fig2


if __name__ == '__main__':
    if len(sys.argv) > 1:
        outdir = sys.argv[1]
        os.makedirs(outdir, exist_ok=True)
        f1, f2 = make_figures()
        f1.savefig(os.path.join(outdir, 'slits_2x2.png'), dpi=150)
        f2.savefig(os.path.join(outdir, 'slits_profile.png'), dpi=150)
        print('已保存：slits_2x2.png, slits_profile.png ->', outdir)
    else:
        make_figures()
        plt.show()
